In [1]:
# import json
from pathlib import Path
import numpy as np
# import pytest
# import advanced_link_skdsp_v3_tx_flexible as txmod
import generate_tx_iq_dataset as genmod
import load_tx_iq_data as loadmod
from tx_controller_tone_pulse_stft_varlen_9 import (
    TonePulseTXControlNetVarLen,
    build_controlled_tone_pulse_batch_from_iq_batches,
)
import torch
import math
import score_iq_decode as scorer
import argparse
import json
import tempfile
from pathlib import Path
from typing import Optional
import advanced_link_skdsp_v7_robust as link7
import os
import torch.nn as nn
import torchaudio.functional as AF




In [2]:
# --- Example: visualize STFT of IQ output stored in .npy ---
import matplotlib.pyplot as plt


def visualize_iq_stft(iq_file, sample_rate_hz, n_fft=256, hop_length=64, win_length=256):
    iq = np.load(iq_file)
    iq_t = torch.as_tensor(iq, dtype=torch.complex64)

    stft = torch.stft(
        iq_t,
        n_fft=n_fft,
        hop_length=hop_length,
        win_length=win_length,
        return_complex=True,
        window=torch.hann_window(win_length),
    )
    mag_db = 20 * torch.log10(torch.clamp(torch.abs(stft), min=1e-8))

    extent = [0, iq_t.numel() / sample_rate_hz, -sample_rate_hz / 2, sample_rate_hz / 2]

    plt.figure(figsize=(12, 4))
    plt.imshow(torch.fft.fftshift(mag_db, dim=0).cpu().numpy(), aspect="auto", origin="lower", extent=extent, cmap="magma")
    plt.colorbar(label="Magnitude (dB)")
    plt.xlabel("Time (s)")
    plt.ylabel("Frequency (Hz)")
    plt.title(f"STFT: {iq_file}")
    plt.tight_layout()
    plt.show()


# Example usage:
# visualize_iq_stft("checkpoints/example_tx_iq.npy", sample_rate_hz=2e9)



In [3]:
import os
from dask import delayed, compute
from dask.diagnostics import ProgressBar

train_path_dat = Path("C:/Users/theon/Jamming/train_set_00")
test_path_dat = Path("C:/Users/theon/Jamming/test_set_00")


def _generate_one_dataset(path_dat: Path, seed: int):
    out_root = path_dat / "dataset"
    return genmod.generate_dataset(
        output_root=out_root,
        num_outputs=250,
        no_jammer_samples_required_per_section=200_000,
        num_sections=3,
        seed=seed,
        random_payload_probability=0.0,
        max_attempts_per_sample=10,
        start_index=-1,
    )


jobs = [
    delayed(_generate_one_dataset)(train_path_dat, seed=3),
    delayed(_generate_one_dataset)(test_path_dat, seed=4),
]

# In notebooks on Windows, process pools frequently fail with BrokenProcessPool.
scheduler = "threads" if os.name == "nt" else "processes"

with ProgressBar():
    try:
        produced_train, produced_test = compute(*jobs, scheduler=scheduler)
    except Exception as exc:
        if scheduler == "processes":
            print(f"Process scheduler failed ({exc}); retrying with threads.")
            produced_train, produced_test = compute(*jobs, scheduler="threads")
        else:
            raise

produced = {
    "train": produced_train,
    "test": produced_test,
}


[                                        ] | 0% Completed | 304.57 ms

C:\Users\theon\Jamming\Conserve\advanced_link_skdsp_v7_robust.py:294: UserWarning: ComplexHalf support is experimental and many operators don't support it yet. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\EmptyTensor.cpp:55.)
  ).to(dtype=DEFAULT_COMPLEX_DTYPE)


[                                        ] | 0% Completed | 1.84 s mssucessfully generated data
sucessfully generated data
[                                        ] | 0% Completed | 9.82 ssucessfully generated data
[                                        ] | 0% Completed | 10.11 ssucessfully generated data
[                                        ] | 0% Completed | 23.22 ssucessfully generated data
sucessfully generated data
[                                        ] | 0% Completed | 24.35 ssucessfully generated data
[                                        ] | 0% Completed | 27.33 ssucessfully generated data
[                                        ] | 0% Completed | 30.34 ssucessfully generated data
[                                        ] | 0% Completed | 33.29 ssucessfully generated data
[                                        ] | 0% Completed | 47.00 ssucessfully generated data
[                                        ] | 0% Completed | 47.53 ssucessfully generated data
suces

In [4]:
test_path = Path("C:/Users/theon/Jamming/val_set_00")
out_root = test_path / "dataset"
produced = genmod.generate_dataset(
                                    output_root=out_root,
                                    num_outputs=500,
                                    min_total_samples=4000,
                                    max_total_samples=5500,
                                    section_len=1024,
                                    num_sections=3,
                                    seed=3,
                                    random_payload_probability=0.0,
                                    start_index=500
                                    )

TypeError: generate_dataset() got an unexpected keyword argument 'min_total_samples'

In [2]:
train_path = Path("C:/Users/theon/Jamming/test_set_00")
# test_path = Path("test_set_00")
# val_path = Path("val_set_00")

# for _ in [train_path, test_path, val_path]:
out_root = train_path / "dataset"
produced = genmod.generate_dataset(
                                    output_root=out_root,
                                    num_outputs=25,
                                    min_total_samples=5000,
                                    max_total_samples=8000,
                                    section_len=1024,
                                    num_sections=3,
                                    seed=3,
                                    random_payload_probability=0.0,
                                    )

sucessfully generated data
sucessfully generated data
sucessfully generated data
sucessfully generated data
sucessfully generated data
sucessfully generated data
sucessfully generated data
sucessfully generated data
sucessfully generated data
sucessfully generated data
sucessfully generated data
sucessfully generated data
sucessfully generated data
sucessfully generated data
sucessfully generated data
sucessfully generated data
sucessfully generated data
sucessfully generated data
sucessfully generated data
sucessfully generated data
sucessfully generated data
sucessfully generated data
sucessfully generated data
sucessfully generated data
sucessfully generated data


In [6]:
train_path_dat = Path("C:/Users/theon/Jamming/train_set_00/dataset")
test_path_dat = Path("C:/Users/theon/Jamming/test_set_00/dataset")

jammer_sampling_freq = 2e9

for path_dat in [train_path_dat, test_path_dat]:
    sample_dirs = loadmod.list_sample_dirs(path_dat)
    
    for sdir in sample_dirs:
        
        sections = loadmod.load_sections(sdir)
        whole = loadmod.load_whole_iq(sdir)
        
        iq1 = link7.resample_iq(sections['sections'][0], whole['meta']['sample_rate_hz'], jammer_sampling_freq)[:1024]
        iq2 = link7.resample_iq(sections['sections'][1], whole['meta']['sample_rate_hz'], jammer_sampling_freq)[:1024]
        iq3 = link7.resample_iq(sections['sections'][2], whole['meta']['sample_rate_hz'], jammer_sampling_freq)[:1024]
    
        iqs = [iq1.numel() != 1024,
               iq2.numel() != 1024,
               iq3.numel() != 1024]
    
        if any(iqs):
            print(f'sdir {sdir} is short iq1.shape : {iq1.shape}  iq2.shape : {iq2.shape}  iq3.shape : {iq3.shape}')

sdir C:\Users\theon\Jamming\train_set_00\dataset\sample_000000 is short iq1.shape : torch.Size([410])  iq2.shape : torch.Size([410])  iq3.shape : torch.Size([410])
sdir C:\Users\theon\Jamming\train_set_00\dataset\sample_000008 is short iq1.shape : torch.Size([410])  iq2.shape : torch.Size([410])  iq3.shape : torch.Size([410])
sdir C:\Users\theon\Jamming\train_set_00\dataset\sample_000011 is short iq1.shape : torch.Size([410])  iq2.shape : torch.Size([410])  iq3.shape : torch.Size([410])
sdir C:\Users\theon\Jamming\train_set_00\dataset\sample_000017 is short iq1.shape : torch.Size([410])  iq2.shape : torch.Size([410])  iq3.shape : torch.Size([410])
sdir C:\Users\theon\Jamming\train_set_00\dataset\sample_000024 is short iq1.shape : torch.Size([410])  iq2.shape : torch.Size([410])  iq3.shape : torch.Size([410])
sdir C:\Users\theon\Jamming\train_set_00\dataset\sample_000033 is short iq1.shape : torch.Size([410])  iq2.shape : torch.Size([410])  iq3.shape : torch.Size([410])
sdir C:\Users\th

In [7]:
problem_cuts = 'C:\\Users\\theon\\Jamming\\train_set_00\\dataset\\sample_000000'

sections = loadmod.load_sections(problem_cuts)
whole = loadmod.load_whole_iq(problem_cuts)
sections['sections'][0].shape

(1024,)

In [8]:
whole['meta']

{'payload_source': 'message',
 'message_length': 776,
 'message': '8YtDt XbiufMdI8X2Y4rUmer BH3M1 XS0DRdJuPnBIKpi99lSi0tcL21pffWQJ EeNajnez0LHtfROUrWW6 Xn I3EM3H MRb1OcWrhQ7TTJ chcVG6MOwUxOVHMWndqN CIEPx3mnPQC4vkRB5ICpeyOxJRkSq1L I7S1L10e0tza9 3Ce6 KRDiKpFfe z3gb9pv MEc0goRqG9hTCzppvEJqa ZgIFI 2g8Pahqfpgi9el   OuOjSXXMUHyQ2pqaWkwfV6Wf3gV O116cFBIj2C2qdPVHp7pWnOna8sEXflmWwdRpdo9KMle EnmhPxjExF6YGVYS1kW E0u1 9tZtum 9 4xrIzs ODL0IBNcI9WzwUEP9s19A7d9jZf7DEiBGEyHrxeGvfOx2lkplHLeT5RadQQ3W jA YqOpJj3o4GmVV5LHnRo  ThLxtwV6pnsQ1Mx69NwinxYTmIIXgrf9 IF TQZ5iT otImooxy1YqsYyvwzGVLd40XON M9dyanD w6zyBe 4oKtr7lgdUD j cRPQSrkekRAiz3C Onf0jzuY 8i2A Mc76Z4x6eGUV5UZCaAHVs6yuAcvZ vdrov4 xhcZ5O0egEZfY dCEmX8yvQoSpgLJ7M FIdRSOlh3landlcv e9gy Qz9R9SeWNYlLx0o XQZwXTxU14D49SIv X ftvc7lmOEhg57QVajyZnRNo5kAEgtsboDKAC 1 Oy7wkfodmzGkn7ZCn SZ5oL4WAoa7MjRSy jU2',
 'random_bits': None,
 'random_seed': None,
 'payload_len': 776,
 'payload_len_bytes': 776,
 'target_num_samples': None,
 'actual_num_samples': 123040,
 '

In [2]:
# "C:\Users\theon\Jamming\train_set_00"
# train_path_dat = Path("C:/Users/theon/Jamming/test_set_00/dataset")
train_path_dat = Path("C:/Users/theon/Jamming/train_set_00/dataset")
sample_dirs = loadmod.list_sample_dirs(train_path_dat)
sdir = sample_dirs[0]
whole = loadmod.load_whole_iq(sdir)
# for sdir in sample_dirs:
#     assert (sdir / "whole_iq.npy").exists()
#     assert (sdir / "whole_meta.json").exists()
#     assert (sdir / "sections.npy").exists()
#     assert (sdir / "sections_meta.json").exists()

# whole = loadmod.load_whole_iq(sdir)
# sections = loadmod.load_sections(sdir)
sdir

WindowsPath('C:/Users/theon/Jamming/train_set_00/dataset/sample_000000')

In [3]:
whole["meta"]

{'payload_source': 'message',
 'message_length': 776,
 'message': '8YtDt XbiufMdI8X2Y4rUmer BH3M1 XS0DRdJuPnBIKpi99lSi0tcL21pffWQJ EeNajnez0LHtfROUrWW6 Xn I3EM3H MRb1OcWrhQ7TTJ chcVG6MOwUxOVHMWndqN CIEPx3mnPQC4vkRB5ICpeyOxJRkSq1L I7S1L10e0tza9 3Ce6 KRDiKpFfe z3gb9pv MEc0goRqG9hTCzppvEJqa ZgIFI 2g8Pahqfpgi9el   OuOjSXXMUHyQ2pqaWkwfV6Wf3gV O116cFBIj2C2qdPVHp7pWnOna8sEXflmWwdRpdo9KMle EnmhPxjExF6YGVYS1kW E0u1 9tZtum 9 4xrIzs ODL0IBNcI9WzwUEP9s19A7d9jZf7DEiBGEyHrxeGvfOx2lkplHLeT5RadQQ3W jA YqOpJj3o4GmVV5LHnRo  ThLxtwV6pnsQ1Mx69NwinxYTmIIXgrf9 IF TQZ5iT otImooxy1YqsYyvwzGVLd40XON M9dyanD w6zyBe 4oKtr7lgdUD j cRPQSrkekRAiz3C Onf0jzuY 8i2A Mc76Z4x6eGUV5UZCaAHVs6yuAcvZ vdrov4 xhcZ5O0egEZfY dCEmX8yvQoSpgLJ7M FIdRSOlh3landlcv e9gy Qz9R9SeWNYlLx0o XQZwXTxU14D49SIv X ftvc7lmOEhg57QVajyZnRNo5kAEgtsboDKAC 1 Oy7wkfodmzGkn7ZCn SZ5oL4WAoa7MjRSy jU2',
 'random_bits': None,
 'random_seed': None,
 'payload_len': 776,
 'payload_len_bytes': 776,
 'target_num_samples': None,
 'actual_num_samples': 123040,
 '

In [5]:
whole = loadmod.load_whole_iq(sdir)
        
whole_iq = torch.tensor(whole['iq'])

genmod._decode_candidate_with_v4(whole_iq, whole["meta"])

{'payload_bytes': b'8YtDt XbiufMdI8X2Y4rUmer BH3M1 XS0DRdJuPnBIKpi99lSi0tcL21pffWQJ EeNajnez0LHtfROUrWW6 Xn I3EM3H MRb1OcWrhQ7TTJ chcVG6MOwUxOVHMWndqN CIEPx3mnPQC4vkRB5ICpeyOxJRkSq1L I7S1L10e0tza9 3Ce6 KRDiKpFfe z3gb9pv MEc0goRqG9hTCzppvEJqa ZgIFI 2g8Pahqfpgi9el   OuOjSXXMUHyQ2pqaWkwfV6Wf3gV O116cFBIj2C2qdPVHp7pWnOna8sEXflmWwdRpdo9KMle EnmhPxjExF6YGVYS1kW E0u1 9tZtum 9 4xrIzs ODL0IBNcI9WzwUEP9s19A7d9jZf7DEiBGEyHrxeGvfOx2lkplHLeT5RadQQ3W jA YqOpJj3o4GmVV5LHnRo  ThLxtwV6pnsQ1Mx69NwinxYTmIIXgrf9 IF TQZ5iT otImooxy1YqsYyvwzGVLd40XON M9dyanD w6zyBe 4oKtr7lgdUD j cRPQSrkekRAiz3C Onf0jzuY 8i2A Mc76Z4x6eGUV5UZCaAHVs6yuAcvZ vdrov4 xhcZ5O0egEZfY dCEmX8yvQoSpgLJ7M FIdRSOlh3landlcv e9gy Qz9R9SeWNYlLx0o XQZwXTxU14D49SIv X ftvc7lmOEhg57QVajyZnRNo5kAEgtsboDKAC 1 Oy7wkfodmzGkn7ZCn SZ5oL4WAoa7MjRSy jU2',
 'payload_len': 776,
 'message': '8YtDt XbiufMdI8X2Y4rUmer BH3M1 XS0DRdJuPnBIKpi99lSi0tcL21pffWQJ EeNajnez0LHtfROUrWW6 Xn I3EM3H MRb1OcWrhQ7TTJ chcVG6MOwUxOVHMWndqN CIEPx3mnPQC4vkRB5ICpeyOxJRkSq1L I7S1

In [7]:
whole = loadmod.load_whole_iq(sdir)
        
whole_iq = whole['iq']
rx_result_check = link7.rx_command_iq(whole_iq, whole["meta"])
score_check = scorer.score_decode(rx_result_check, whole["meta"])
score_check

1.0

In [8]:
rx_result_check

{'payload_bytes': b'8YtDt XbiufMdI8X2Y4rUmer BH3M1 XS0DRdJuPnBIKpi99lSi0tcL21pffWQJ EeNajnez0LHtfROUrWW6 Xn I3EM3H MRb1OcWrhQ7TTJ chcVG6MOwUxOVHMWndqN CIEPx3mnPQC4vkRB5ICpeyOxJRkSq1L I7S1L10e0tza9 3Ce6 KRDiKpFfe z3gb9pv MEc0goRqG9hTCzppvEJqa ZgIFI 2g8Pahqfpgi9el   OuOjSXXMUHyQ2pqaWkwfV6Wf3gV O116cFBIj2C2qdPVHp7pWnOna8sEXflmWwdRpdo9KMle EnmhPxjExF6YGVYS1kW E0u1 9tZtum 9 4xrIzs ODL0IBNcI9WzwUEP9s19A7d9jZf7DEiBGEyHrxeGvfOx2lkplHLeT5RadQQ3W jA YqOpJj3o4GmVV5LHnRo  ThLxtwV6pnsQ1Mx69NwinxYTmIIXgrf9 IF TQZ5iT otImooxy1YqsYyvwzGVLd40XON M9dyanD w6zyBe 4oKtr7lgdUD j cRPQSrkekRAiz3C Onf0jzuY 8i2A Mc76Z4x6eGUV5UZCaAHVs6yuAcvZ vdrov4 xhcZ5O0egEZfY dCEmX8yvQoSpgLJ7M FIdRSOlh3landlcv e9gy Qz9R9SeWNYlLx0o XQZwXTxU14D49SIv X ftvc7lmOEhg57QVajyZnRNo5kAEgtsboDKAC 1 Oy7wkfodmzGkn7ZCn SZ5oL4WAoa7MjRSy jU2',
 'payload_len': 776,
 'message': '8YtDt XbiufMdI8X2Y4rUmer BH3M1 XS0DRdJuPnBIKpi99lSi0tcL21pffWQJ EeNajnez0LHtfROUrWW6 Xn I3EM3H MRb1OcWrhQ7TTJ chcVG6MOwUxOVHMWndqN CIEPx3mnPQC4vkRB5ICpeyOxJRkSq1L I7S1

In [4]:
whole = loadmod.load_whole_iq(sdir)
        
whole_iq = whole['iq']
rx_result_check = link4.rx_command_iq(whole_iq, whole["meta"])
score_check = scorer.score_decode(rx_result_check, whole["meta"])
score_check

from loop best_start : 0, type : <class 'int'>
from loop best_cfo : 0.0, type : <class 'float'>
from loop best_metric : 1157.6636962890625, type : <class 'float'>
coarse_start: 0, coarse_cfo_hz : 0.0, coarse_metric : 1157.6636962890625


1.0

In [6]:
whole = loadmod.load_whole_iq(sdir)
        
whole_iq = whole['iq']
rx_result_check = link_numpy.rx_command_iq(whole_iq, whole["meta"])
score_check = scorer.score_decode(rx_result_check, whole["meta"])
score_check

1.0

In [9]:
whole = loadmod.load_whole_iq(sdir)
        
whole_iq = whole['iq']
rx_result_check = link_torch.rx_command_iq(whole_iq, whole["meta"])
score_check = scorer.score_decode(rx_result_check, whole["meta"])
score_check

RuntimeError: Input type (CUDAComplexDoubleType) and weight type (CUDAComplexFloatType) should be the same

In [11]:
link_torch.bpsk_map([1,0])

tensor([ 1.+0.j, -1.+0.j], device='cuda:0')

In [13]:
link_numpy.bpsk_map([1,0,1,0])

array([ 1.+0.j, -1.+0.j,  1.+0.j, -1.+0.j], dtype=complex64)

In [14]:
link_torch.tx_waveform(bits = [1.0], sps = 8, beta = 0.35, span = 6)

tensor([-0.0021+0.j, -0.0022+0.j, -0.0016+0.j, -0.0005+0.j,  0.0010+0.j,  0.0023+0.j,
         0.0032+0.j,  0.0034+0.j], device='cuda:0')

In [15]:
link_numpy.tx_waveform(bits = [1.0], sps = 8, beta = 0.35, span = 6)

array([-0.00207092+0.j, -0.0021541 +0.j, -0.00158815+0.j, -0.00046041+0.j,
        0.00097236+0.j,  0.00233953+0.j,  0.00324914+0.j,  0.00339539+0.j],
      dtype=complex64)

In [6]:
whole = loadmod.load_whole_iq(sdir)
        
whole_iq = whole['iq']
rx_result_check = link_numpy.rx_command_iq(whole_iq, whole["meta"])
score_check = scorer.score_decode(rx_result_check, whole["meta"])
score_check

RuntimeError: No valid packet found after acquisition, header decode, FEC decode, and CRC

In [6]:
sdir

WindowsPath('C:/Users/theon/Jamming/train_set_00/dataset/sample_000004')

In [4]:
iq_path = Path(sdir / "whole_iq.npy")
meta_path = Path(sdir / "whole_meta.json")
print(f'sdir : {sdir}')
# assert (sdir / "whole_iq.npy").exists()
#     assert (sdir / "whole_meta.json").exists()
               
score = scorer.main(["--iq", str(iq_path), "--metadata", str(meta_path)])
score

sdir : C:\Users\theon\Jamming\train_set_00\dataset\sample_000000
from loop best_start : 1065, type : <class 'int'>
from loop best_cfo : 7500.0, type : <class 'float'>
from loop best_metric : 216.97389221191406, type : <class 'float'>
0.0


0.0

In [5]:
whole = loadmod.load_whole_iq(sdir)
        
whole_iq = whole['iq']
# whole_iq
rx_result_check = link.rx_command_iq(whole_iq, whole["meta"])
# score_check = scorer.score_decode(rx_result_check, whole["meta"])

RuntimeError: No valid packet found after acquisition, header decode, FEC decode, and CRC

In [5]:
sections = loadmod.load_sections(sdir)
sections['sections'][0].shape

(1024,)

In [6]:
whole['iq'].shape[0]

123040

In [19]:
sdir_01 = sample_dirs[1]
whole_01 = loadmod.load_whole_iq(sdir_01)
whole_01['meta']['fec']


'conv'

In [6]:
score_check = scorer.score_decode(rx_result_check, whole_01["meta"])
score_check

1.0

In [3]:
iq_path = Path(sample_dirs[4] / "whole_iq.npy")
meta_path = Path(sample_dirs[4] / "whole_meta.json")
print(f'sdir : {sample_dirs[4]}')
# assert (sdir / "whole_iq.npy").exists()
#     assert (sdir / "whole_meta.json").exists()
               
score = scorer.main(["--iq", str(iq_path), "--metadata", str(meta_path)])
score

sdir : C:\Users\theon\Jamming\train_set_00\dataset\sample_000004
from broadcast best_start : 59017, type : <class 'int'>
from broadcast best_cfo : 11000.0, type : <class 'float'>
from broadcast best_metric : 239.02914428710938, type : <class 'float'>
0.0


0.0

In [22]:
score = JammerLossFunction._evaluate(whole_01['iq'], whole_01)
type(score)

float

In [39]:
jl = JammerLoss()
jt = torch.tensor(whole_01['iq'], device ='cuda', dtype=torch.cfloat)
jt.shape
    

torch.Size([114080])

In [40]:
whole_01['iq'].shape

(114080,)

In [41]:
whole_01['iq']

array([-0.01453683+0.00050056j, -0.00456747+0.00483577j,
       -0.00360805+0.00612219j, ...,  0.37030873-0.07326386j,
        0.36604854-0.06185472j,  0.00041609+0.00297634j],
      shape=(114080,), dtype=complex64)

In [43]:
jt.cpu().numpy().shape


(114080,)

In [23]:
score

1.0

In [9]:
scoring_dict

{0: 100.0}

epoch : 0


C:\Users\theon\AppData\Local\Temp\ipykernel_9948\57984809.py:60: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  jammed = torch.tensor(jammed, device ='cuda', dtype=torch.cfloat).requires_grad_(True)


RuntimeError: expected scalar type Float but found ComplexFloat

In [ ]:
def compute_returns(rewards, gamma: float):
    """
    Compute discounted returns for one episode.
    """
    returns = []
    G = 0.0
    for r in reversed(rewards):
        G = r + gamma * G
        returns.append(G)
    returns.reverse()
    return returns


def train_policy_gradient(
    env,
    policy: nn.Module,
    optimizer: optim.Optimizer,
    num_episodes: int = 1000,
    gamma: float = 0.99,
    device: str = "cpu",
    max_steps_per_episode: int = 1000,
    normalize_returns: bool = True,
    print_every: int = 50,
):
    """
    Generic REINFORCE training loop.

    Args:
        env: Gym-like environment with reset() and step(action)
        policy: PyTorch model mapping observation -> action logits
        optimizer: PyTorch optimizer
        num_episodes: number of training episodes
        gamma: discount factor
        device: torch device
        max_steps_per_episode: safety cap per episode
        normalize_returns: whether to normalize discounted returns
        print_every: logging frequency
    """
    device = torch.device(device)
    policy.to(device)

    reward_history = []

    for episode in range(1, num_episodes + 1):
        # Gymnasium reset API compatibility
        reset_result = env.reset()
        state = reset_result[0] if isinstance(reset_result, tuple) else reset_result

        log_probs = []
        rewards = []
        total_reward = 0.0

        for step in range(max_steps_per_episode):
            action, log_prob = select_action(policy, state, device)

            step_result = env.step(action)
            if len(step_result) == 5:
                next_state, reward, terminated, truncated, _ = step_result
                done = terminated or truncated
            else:
                next_state, reward, done, _ = step_result

            log_probs.append(log_prob)
            rewards.append(reward)
            total_reward += reward
            state = next_state

            if done:
                break

        # Compute discounted returns
        returns = compute_returns(rewards, gamma)
        returns = torch.tensor(returns, dtype=torch.float32, device=device)

        if normalize_returns and len(returns) > 1:
            returns = (returns - returns.mean()) / (returns.std() + 1e-8)

        # Policy gradient loss
        loss = []
        for log_prob, G in zip(log_probs, returns):
            loss.append(-log_prob * G)
        loss = torch.stack(loss).sum()

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        reward_history.append(total_reward)

        if episode % print_every == 0:
            avg_reward = sum(reward_history[-print_every:]) / print_every
            print(
                f"Episode {episode:4d} | "
                f"Avg Reward: {avg_reward:8.3f} | "
                f"Last Reward: {total_reward:8.3f} | "
                f"Loss: {loss.item():8.3f}"
            )

    return reward_history